In [14]:
import json
import numpy as np
import nltk
import torch
import google.generativeai as genai
from sentence_transformers import CrossEncoder
from pathlib import Path
from google.colab import userdata

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
KB_PATH = "/content/drive/MyDrive/CS_510/knowledge_base.json"

In [ ]:
nltk.download('punkt')
nltk.download('punkt_tab')

In [ ]:
!pip install -U -q google-generativeai

In [40]:
GEMINI_KEY = userdata.get('gemini-key')
genai.configure(api_key=GEMINI_KEY)
model = genai.GenerativeModel("gemini-3-flash-preview")

DRIVE_BASE = Path("/content/drive/MyDrive/CS_510")
KB_PATH = DRIVE_BASE / "knowledge_base.json"
OUTPUT_JSON = DRIVE_BASE / "claim_vs_reality_final.json"

# Ranking Model
print("Loading Cross-Encoder for ranking...")
device = "cuda" if torch.cuda.is_available() else "cpu"
ranker = CrossEncoder('cross-encoder/stsb-distilroberta-base', device=device)

with open(KB_PATH, "r", encoding="utf-8") as f:
    kb = json.load(f)

def ask_gemini(prompt):
    """Helper to get concise responses from Gemini."""
    try:
        response = model.generate_content(prompt)
        return response.text.strip().replace("*", "")
    except Exception as e:
        print(f"Gemini Error: {e}")
        return ""

def generate_smart_claim_vs_reality(kb, test_mode=True):
    all_products_output = []

    for product_name, entry in kb.items():
        description = entry.get("combined_description", "")
        reviews = [r.get("body", "") for r in entry.get("all_reviews", [])]

        if not description or not reviews:
            continue

        print(f"Processing: {product_name}")

        claim_sentences = nltk.sent_tokenize(description)
        review_sentences = []
        for r_text in reviews:
            review_sentences.extend(nltk.sent_tokenize(r_text))

        if not review_sentences:
            continue

        claim_vs_reality_rows = []

        for c_sent in claim_sentences[:3]:
            label_prompt = f"Convert this product claim into a 2-3 word title (e.g., 'Fit and comfort'). Claim: {c_sent}"
            feature_label = ask_gemini(label_prompt).title()


            pairs = [[c_sent, r_sent] for r_sent in review_sentences]
            scores = ranker.predict(pairs)


            top_indices = np.argsort(scores)[-5:][::-1]
            top_relevant_sentences = [review_sentences[i] for i in top_indices if scores[i] > 0.15]

            if not top_relevant_sentences:
                actual_summary = "No significant user feedback found."
            else:
                context = " ".join(top_relevant_sentences)
                summary_prompt = f"Summarize these user reviews about '{feature_label}' into one concise sentence: {context}"
                actual_summary = ask_gemini(summary_prompt)

            claim_vs_reality_rows.append({
                "feature": feature_label or "Product Feature",
                "advertised": c_sent,
                "actual": actual_summary
            })

        product_data = {
            "product_name": product_name,
            "claim_vs_reality": claim_vs_reality_rows
        }

        all_products_output.append(product_data)

        if test_mode:
            print("\n--- TEST MODE OUTPUT ---")
            print(json.dumps(product_data, indent=2))
            break

    with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
        json.dump(all_products_output, f, indent=2, ensure_ascii=False)

    print(f"\n Final formatted JSON saved to: {OUTPUT_JSON}")
    return all_products_output

output_data = generate_smart_claim_vs_reality(kb, test_mode=False)

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


Loading Cross-Encoder for ranking...


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: cross-encoder/stsb-distilroberta-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Processing: OxiClean White Revive Laundry Whitener
Processing: Apple Watch Series 11
Processing: Dyson V8 Cordless Vacuum
Processing: Crocs
Processing: Apple Airpods 4
Processing: Mini Mic Pro
Processing: Owala Water Bottle
Processing: Mighty Patch Original (Hero Cosmetics)
Processing: COSRX Snail Mucin 96% Power Repairing Essence
Processing: Bedsure Satin Pillowcase Set
Processing: RX Bar Vanilla Almond Protein Bars
Processing: Byoma Balancing Face Mist
Processing: Milk Makeup Cooling Water Jelly Tint
Processing: Phillip Sonicare 4100 Series Electric Toothbrush
Processing: Yankee Candle Lemon Lavender Scented Candle
Processing: Keurig K-Mini Single Serve Coffee Maker

✅ Final formatted JSON saved to: /content/drive/MyDrive/CS_510/claim_vs_reality_final.json
